In [1]:
# 薬を飲み忘れた翌日の血糖値と血圧　20251214
# Outlook予定表 → 「服薬×の日」の翌日の 血糖値・血圧(収縮期/拡張期/心拍) を df で一覧
# 2025-12-14 置き換え版（丸ごと差し替え）

import re
import datetime as dt
import pandas as pd
import win32com.client as win32

# =========================
# 1) 期間設定（ここだけ編集）
# =========================
start_date = dt.datetime(2023, 1, 1, 0, 0, 0)
end_date   = dt.datetime(2025, 12, 31, 23, 59, 59)

PROFILE_NAME = None  # 既定でよければ None
STORE_HINTS  = ["kishi_isao@outlook.jp"]  # 必要なら変更

# =========================
# 2) 服薬判定ルール（あなたの条件）
#    「服薬」があり、〇/○/◯ が含まれていれば ○、含まれなければ ×
# =========================
MARU_MARKS = ("〇", "○", "◯")

def judge_fukuyaku(subject: str) -> str:
    return "○" if any(m in (subject or "") for m in MARU_MARKS) else "×"

# =========================
# 3) 解析（あなたの記録形式対応）
# 例:
#   朝食前 体重 69.8kg 血糖値 117/8:15 血圧 129-75/61/8:17, ... , 119-72/57/8:21
# 血糖値 → 117
# 血圧   → 最後の1件を採用：119-72/57/8:21 → 収縮期119 拡張期72 心拍57
# =========================
RE_GLU = re.compile(r"血糖値\s*[:：]?\s*(\d{2,3})(?:\s*[\/／]\s*\d{1,2}:\d{2})?")

RE_BP_TRIPLE = re.compile(
    r"(\d{2,3})\s*[-‐-–—ー－]\s*(\d{2,3})\s*[\/／]\s*(\d{2,3})(?:\s*[\/／]\s*\d{1,2}:\d{2})?"
)

def parse_metrics(text: str):
    """text から (血糖値, 収縮期, 拡張期, 心拍数) を抽出。なければ None"""
    text = text or ""
    glu = sys = dia = hr = None

    # 血糖値（最初の1件）
    m = RE_GLU.search(text)
    if m:
        glu = int(m.group(1))

    # 血圧は「血圧」以降を対象にして、最後の1件
    pos = text.find("血圧")
    bp_area = text[pos:] if pos != -1 else text

    matches = RE_BP_TRIPLE.findall(bp_area)
    if matches:
        s, d, p = matches[-1]          # 最後の1件
        sys, dia, hr = int(s), int(d), int(p)

    return glu, sys, dia, hr

# =========================
# 4) Outlook予定表アクセス（あなたの既存構造）
# =========================
def iter_stores(session):
    stores = session.Stores
    for i in range(1, stores.Count + 1):   # 1-based index
        yield stores.Item(i)

def pick_store_by_hints(session, hints):
    if not hints:
        return None
    hints_l = [h.lower() for h in hints]
    for s in iter_stores(session):
        name = (s.DisplayName or "").lower()
        if any(h in name for h in hints_l):
            return s
    return None

def day0(d: dt.datetime) -> dt.datetime:
    """日付(00:00)に丸める"""
    return dt.datetime(d.year, d.month, d.day)

def get_text(appt) -> str:
    """件名+本文を解析対象に（本文に記録していても拾える）"""
    subj = appt.Subject or ""
    body = ""
    try:
        body = appt.Body or ""
    except Exception:
        pass
    return subj + "\n" + body

# ===== Classic Outlook に接続 =====
app = win32.gencache.EnsureDispatch("Outlook.Application")
session = app.GetNamespace("MAPI")
session.Logon(PROFILE_NAME or "", "", False, False)

# ストア決定
store = pick_store_by_hints(session, STORE_HINTS) if STORE_HINTS else None
if store is None:
    store = session.DefaultStore

print(f"使用ストア: {store.DisplayName}")

# ストア内の「予定表」取得（olFolderCalendar=9）
try:
    calendar = store.GetDefaultFolder(9)
except Exception:
    root = store.GetRootFolder()
    for cand in ("予定表", "Calendar", "カレンダー"):
        try:
            calendar = root.Folders[cand]
            break
        except Exception:
            pass
    else:
        raise RuntimeError("このストア内で予定表フォルダーが見つかりませんでした。")

items = calendar.Items
items.IncludeRecurrences = True
items.Sort("[Start]")

restriction = (
    f"[Start] >= '{start_date:%m/%d/%Y %I:%M %p}' AND "
    f"[Start] <= '{end_date:%m/%d/%Y %I:%M %p}'"
)
filtered = items.Restrict(restriction)

# =========================
# 5) 予定表を走査して「日付ごと」に情報を集約
#    - metrics_by_date: その日の血糖/血圧/心拍（同日に複数あっても"最後"を優先）
#    - fuku_by_date:    その日の服薬判定（最初に見つけた服薬予定を採用）
# =========================
metrics_by_date = {}  # date -> {"glu":..,"sys":..,"dia":..,"hr":..}
fuku_by_date = {}     # date -> {"fukuyaku":..,"subject":..}

count = 0
for appt in filtered:
    try:
        _ = appt.Start  # 念のため（取得不能アイテム回避）
        count += 1

        d = day0(appt.Start)
        subject = appt.Subject or ""
        text = get_text(appt)

        # 測定値（その日の最後を採用したいので、取れたら上書きでOK）
        glu, sys, dia, hr = parse_metrics(text)
        if any(v is not None for v in (glu, sys, dia, hr)):
            rec = metrics_by_date.get(d, {"glu": None, "sys": None, "dia": None, "hr": None})
            # 血糖値は取れたら上書き（後の予定があればそれを優先）
            if glu is not None: rec["glu"] = glu
            # 血圧は「最後の1件」採用なので、取れたら上書き
            if sys is not None: rec["sys"] = sys
            if dia is not None: rec["dia"] = dia
            if hr  is not None: rec["hr"]  = hr
            metrics_by_date[d] = rec

        # 服薬判定（件名で判定）
        if "服薬" in subject:
            if d not in fuku_by_date:  # 同日に複数ある場合は最初の1件を採用
                fuku_by_date[d] = {
                    "fukuyaku": judge_fukuyaku(subject),
                    "subject": subject
                }

    except Exception:
        continue

print("予定取得件数(走査):", count)
print("服薬判定日数:", len(fuku_by_date))
print("測定値が取れた日数:", len(metrics_by_date))

# =========================
# 6) 服薬×の日 → 翌日の測定値 を df に
# =========================
rows = []
for d, info in fuku_by_date.items():
    if info["fukuyaku"] != "×":
        continue

    next_d = d + dt.timedelta(days=1)
    rec = metrics_by_date.get(next_d, {"glu": None, "sys": None, "dia": None, "hr": None})

    rows.append({
        "服薬×の日": d,
        "翌日": next_d,
        "翌日_血糖値": rec["glu"],
        "翌日_収縮期": rec["sys"],
        "翌日_拡張期": rec["dia"],
        "翌日_心拍数": rec["hr"],
        "服薬予定件名": info["subject"],
    })

df = pd.DataFrame(rows).sort_values("服薬×の日").reset_index(drop=True)

# =========================
# 7) 平均値行を追加（欠損は除外して平均）
# =========================
avg_cols = ["翌日_血糖値", "翌日_収縮期", "翌日_拡張期", "翌日_心拍数"]

# 数値化（文字になっていても平均できるように）
for c in avg_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

avg = df[avg_cols].mean(numeric_only=True)

avg_row = {
    "服薬×の日": "平均",
    "翌日": "",
    "翌日_血糖値": avg["翌日_血糖値"],
    "翌日_収縮期": avg["翌日_収縮期"],
    "翌日_拡張期": avg["翌日_拡張期"],
    "翌日_心拍数": avg["翌日_心拍数"],
    "服薬予定件名": f"平均（n={df['翌日_血糖値'].notna().sum()}など列ごとに欠損除外）",
}

df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)

# 表示用に小数1桁（血圧/心拍は整数のままでもOK。必要なら変えてください）
df



使用ストア: kishi_isao@outlook.jp
予定取得件数(走査): 10417
服薬判定日数: 1041
測定値が取れた日数: 1041


,服薬×の日,翌日,翌日_血糖値,翌日_収縮期,翌日_拡張期,翌日_心拍数,服薬予定件名
0,2023-10-29 00:00:00,2023-10-30 00:00:00,130.0,145.0,74.0,49.0,服薬☓
1,2024-04-03 00:00:00,2024-04-04 00:00:00,138.0,142.0,92.0,65.0,服薬△
2,2024-04-04 00:00:00,2024-04-05 00:00:00,126.0,136.0,75.0,71.0,服薬△
3,2024-04-11 00:00:00,2024-04-12 00:00:00,160.0,151.0,90.0,53.0,服薬☓
4,2024-09-02 00:00:00,2024-09-03 00:00:00,138.0,124.0,76.0,58.0,服薬☓
5,2024-09-15 00:00:00,2024-09-16 00:00:00,105.0,111.0,71.0,51.0,服薬
6,2024-12-05 00:00:00,2024-12-06 00:00:00,NaN,NaN,NaN,NaN,服薬☓
7,2024-12-06 00:00:00,2024-12-07 00:00:00,NaN,NaN,NaN,NaN,服薬☓
8,2024-12-08 00:00:00,2024-12-09 00:00:00,143.0,126.0,74.0,54.0,服薬×
9,2025-03-19 00:00:00,2025-03-20 00:00:00,115.0,141.0,75.0,51.0,服薬×


In [8]:
# ===== ここから（追加セル：服薬○=〇/○/◯/たのマル を含む） =====

# 1) 「服薬あり」を示す表記（必要ならここに追加）
MARU_MARKS = ("〇", "○", "◯", "たのマル")

def is_fukuyaku_ok(subject: str) -> bool:
    s = subject or ""
    return ("服薬" in s) and any(m in s for m in MARU_MARKS)

# 2) fuku_by_date を「○/×」で再判定（既存の判定を上書きして整合を取る）
#    ※ fuku_by_date: {date: {"fukuyaku":..., "subject":...}} を想定
for d, info in list(fuku_by_date.items()):
    subj = info.get("subject", "")
    if "服薬" in (subj or ""):
        info["fukuyaku"] = "○" if any(m in subj for m in MARU_MARKS) else "×"
        fuku_by_date[d] = info

# 3) 服薬○の日 → 翌日の測定値 df を作成
rows_o = []
for d, info in fuku_by_date.items():
    if info.get("fukuyaku") != "○":
        continue
    next_d = d + dt.timedelta(days=1)
    rec = metrics_by_date.get(next_d, {"glu": None, "sys": None, "dia": None, "hr": None})
    rows_o.append({
        "服薬の日": d,
        "翌日": next_d,
        "翌日_血糖値": rec["glu"],
        "翌日_収縮期": rec["sys"],
        "翌日_拡張期": rec["dia"],
        "翌日_心拍数": rec["hr"],
    })
df_o = pd.DataFrame(rows_o).sort_values("服薬の日").reset_index(drop=True)

# 4) 服薬×の df（すでに df がある前提だが、念のため再作成も可能に）
#    ここでは既存 df をそのまま使用します（列名が同じ想定）
df_x = df.copy()

# 5) 平均・n・差分（×−○）の比較表を作成
avg_cols = ["翌日_血糖値", "翌日_収縮期", "翌日_拡張期", "翌日_心拍数"]

def summarize(df_src, label):
    df_num = df_src[avg_cols].apply(pd.to_numeric, errors="coerce")
    avg = df_num.mean()
    n   = df_num.notna().sum()
    return {
        "区分": label,
        "血糖値_平均": avg["翌日_血糖値"], "血糖値_n": int(n["翌日_血糖値"]),
        "収縮期_平均": avg["翌日_収縮期"], "収縮期_n": int(n["翌日_収縮期"]),
        "拡張期_平均": avg["翌日_拡張期"], "拡張期_n": int(n["翌日_拡張期"]),
        "心拍数_平均": avg["翌日_心拍数"], "心拍数_n": int(n["翌日_心拍数"]),
    }

summary_o = summarize(df_o, "服薬○（〇/○/◯/たのマル）")
summary_x = summarize(df_x, "服薬×")

diff = {
    "区分": "差分（× − ○）",
    "血糖値_平均": summary_x["血糖値_平均"] - summary_o["血糖値_平均"], "血糖値_n": "",
    "収縮期_平均": summary_x["収縮期_平均"] - summary_o["収縮期_平均"], "収縮期_n": "",
    "拡張期_平均": summary_x["拡張期_平均"] - summary_o["拡張期_平均"], "拡張期_n": "",
    "心拍数_平均": summary_x["心拍数_平均"] - summary_o["心拍数_平均"], "心拍数_n": "",
}

df_compare = pd.DataFrame([summary_o, summary_x, diff])

# 表示（2つ出します）
print("=== 服薬○ の翌日一覧（先頭） ===")
display(df_o.head(20))

print("=== ○ vs × 比較（平均・n・差分） ===")
display(df_compare)

# ===== ここまで =====

=== 服薬○ の翌日一覧（先頭） ===


,服薬の日,翌日,翌日_血糖値,翌日_収縮期,翌日_拡張期,翌日_心拍数
0,2023-01-02,2023-01-03,119.0,125.0,70.0,49.0
1,2023-01-03,2023-01-04,119.0,140.0,77.0,48.0
2,2023-01-04,2023-01-05,101.0,131.0,76.0,53.0
3,2023-01-05,2023-01-06,121.0,137.0,78.0,52.0
4,2023-01-06,2023-01-07,117.0,130.0,70.0,55.0
5,2023-01-07,2023-01-08,125.0,109.0,75.0,60.0
6,2023-01-08,2023-01-09,153.0,123.0,74.0,51.0
7,2023-01-09,2023-01-10,148.0,119.0,71.0,51.0
8,2023-01-10,2023-01-11,127.0,114.0,60.0,83.0
9,2023-01-11,2023-01-12,132.0,NaN,NaN,NaN


=== ○ vs × 比較（平均・n・差分） ===


,区分,血糖値_平均,血糖値_n,収縮期_平均,収縮期_n,拡張期_平均,拡張期_n,心拍数_平均,心拍数_n
0,服薬○（〇/○/◯/たのマル）,123.813696,993,120.292261,982,70.538697,982,54.985743,982
1,服薬×,130.400000,11,138.400000,11,77.700000,11,55.100000,11
2,差分（× − ○）,6.586304,,18.107739,,7.161303,,0.114257,
